# soundstream demo

drop an audio url in the cell below, hit run all, you get a reconstruction at the bottom. default points to a clip from lj speech (the one referenced in the assignment). swap it for whatever — i downmix to mono and resample to 16 khz on the way in anyway.

full repo at https://github.com/tolyaho/NeuralAudioCodec.

## setup

clones my repo, installs `soundfile` (colab already has the rest), and pulls my checkpoint from hugging face. safe to re-run — clone gets reused, checkpoint download skips if it's already on disk.

In [ ]:
import os
if not os.path.isdir('NeuralAudioCodec'):
    !git clone -q https://github.com/tolyaho/NeuralAudioCodec.git
%cd NeuralAudioCodec

In [ ]:
!pip install -q soundfile

In [ ]:
!bash scripts/download_checkpoint.sh

## audio in

whatever you put in `AUDIO_URL` goes through the codec. wav works as is; for mp3 or m4a you'll need ffmpeg, so drop `!apt-get install -y ffmpeg` above this cell. the cell below pulls the file, mixes to mono, resamples to 16 khz — that's the only thing the codec knows since that's what i trained on.

In [ ]:
AUDIO_URL = 'https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav'

In [ ]:
import io
import requests
import torch
import torchaudio

SAMPLE_RATE = 16000

response = requests.get(AUDIO_URL, timeout=30)
response.raise_for_status()

waveform, sr = torchaudio.load(io.BytesIO(response.content))

if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
if sr != SAMPLE_RATE:
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

print(f'loaded {waveform.shape[-1] / SAMPLE_RATE:.2f} s, {SAMPLE_RATE} Hz, mono')

## codec

builds the model and loads my checkpoint. constructor args have to match what i used during training or the state dict won't load — i keep them in the hydra config, hardcoded them here so the demo doesn't drag any of that in.

the ema-buffer fixup at the bottom is a leftover from a resume bug i hit early on — some old checkpoints didn't save those buffers and the model silently broke at quantization. the final checkpoint does save them so it's a no-op now, but i'd rather keep the safety net than rip it out and forget about it later.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.model.soundstream import SoundStream

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SoundStream(
    in_channels=1,
    base_channels=32,
    latent_dim=512,
    codebook_size=1024,
    num_quantizers=8,
).to(device).eval()

ckpt = torch.load('checkpoints/final_soundstream.pt', map_location=device)
state = ckpt['codec'] if 'codec' in ckpt else ckpt.get('model', ckpt)
missing, _ = model.load_state_dict(state, strict=False)

if any('ema_' in k for k in missing):
    for vq in model.quantizer.quantizers:
        vq.ema_cluster_sum.data.copy_(vq.codebook.weight.data)
        vq.ema_cluster_size.fill_(1.0)

print('loaded checkpoint at step', ckpt.get('step'))

## forward pass

runs the whole clip through encoder → quantizer → decoder in one go, no chunking. i clamp the output back into `[-1, 1]` after the decoder because it occasionally pushes a sample slightly past that on sharp onsets, which is enough to make the audio widget complain.

In [ ]:
with torch.no_grad():
    audio = waveform.unsqueeze(0).to(device)  # [1, 1, T]
    out = model(audio)
    reconstruction = out['reconstruction'].clamp(-1.0, 1.0)

print('input ', tuple(audio.shape))
print('output', tuple(reconstruction.shape))

## listen

original on top, reconstruction below. if it sounds like the same person saying the same thing with a touch of high-end softening, that's about right — codec runs at a fairly low bitrate (6.4 kbps) and the upper band is the first thing to take a hit.

In [ ]:
from IPython.display import Audio, display

original = audio.squeeze().cpu().numpy()
reconstructed = reconstruction.squeeze().cpu().numpy()

print('original')
display(Audio(original, rate=SAMPLE_RATE))
print('reconstructed')
display(Audio(reconstructed, rate=SAMPLE_RATE))